# 18 — Anotación, Fase 2: doble anotación y adjudicación de las series de control

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`04-jss-anotacion-f1-calibracion.ipynb`](04-jss-anotacion-f1-calibracion.ipynb)
(Fase 1: método, medidas y vacíos de la guía) · [manual de anotación](../docs/anotacion/manual-anotacion-oe1.md)

**Estado: en curso.** Hoy cubre **ES2016** (§2) e **IS1003** (§3), dos de las cuatro series de
control. Las secciones de TS3005 y ES2008 se agregan a medida que cada serie se adjudica (§4), y
el resumen entre series (§5) se completa con la última.

---

> **Aviso de alcance (ADR 0005).** ES2016 e IS1003 son series de **evaluación**. Este notebook
> muestra solo **estadísticas agregadas**: conteos, tasas, kappas, tablas de contingencia de
> etiquetas e identificadores de decisión y de par. **No muestra ni cita** frases de decisión,
> evidencias, transcripciones, objetos, contenidos, notas ni razones de adjudicación. Las
> celdas seleccionan columnas explícitas y fallan si se les pide una columna de contenido (§1).
>
> **El kappa de estado es diagnóstico y NO se reporta en la tesis** (manual §6;
> [`tareas-jss.md` §2.3](../docs/anotacion/asignacion/tareas-jss.md)). Los kappas reportables
> de OE1 son los de la Tarea D (existencia, tipo y dirección de los pares), que se calculan
> sobre pocos pares por serie (4 en ES2016, 8 en IS1003) y deben leerse con ese límite.

## Qué hace este notebook

Es el registro de la **Fase 2** de la anotación humana de OE1: `gm` y `gv` etiquetan por
separado cada serie de control y `jss` adjudica los desacuerdos. Para cada serie adjudicada
(ES2016 e IS1003) reconstruye:

1. Qué insumos entran y cómo se protege el contenido de evaluación (§1).
2. Cuánto coincidieron los anotadores en decisiones (Tarea A) y en pares (Tarea B), cómo quedó la
   referencia adjudicada y qué patrón tuvieron los desacuerdos (§2).
3. Cómo se compara cada serie con las de la Fase 1 y qué implica para las series siguientes
   (§2.6, §2.7, §3.6, §3.7).
4. Dónde se agregan las series que faltan y el resumen entre series (§4, §5).

## Qué NO hace este notebook

- **No muestra contenido de evaluación** (ADR 0005).
- **No modifica ninguna etiqueta.** Lee los CSV versionados; las etiquetas son humanas.
- **No calcula en el notebook.** Toda cifra viene de una función testeada de
  `src/afg/annotation/` ([`CONTRIBUTING.md` §7.1](../CONTRIBUTING.md)); aquí solo se importa,
  se llama, se agrupa por etiqueta y se explica.

In [1]:
import warnings

import pandas as pd

from afg.annotation.adjudication import validate_adjudication
from afg.annotation.agreement import (
    annotator_csvs,
    compute_series_agreement,
    series_axis_profiles,
)
from afg.annotation.calibration import (
    adjudicated_pairs,
    annotator_status_distribution,
    chance_agreement,
    compuesta_children,
    decision_agreement,
    final_status_counts,
    resolved_disagreements,
    side_taken_counts,
    status_contingency,
)
from afg.annotation.workspace import TaskKind, WorkspacePaths
from afg.shared.paths import PROJECT_ROOT

# Phase 2 series already adjudicated. Append a series here when its adjudication is merged.
F2_SERIES = ["ES2016", "IS1003"]
SERIES = "ES2016"  # section 2
SERIES_IS = "IS1003"  # section 3
# Phase 1 series, used only as the comparison baseline (they are development series).
F1_SERIES = ["ES2015", "TS3009"]

# Cohen's kappa is undefined (NaN) when both annotators use a single label; scikit-learn warns
# about it on every such axis. The NaN is explained in section 2.5, so the warnings are muted.
warnings.filterwarnings("ignore", module="sklearn")

PATHS = WorkspacePaths()  # default locations under data/processed/

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

# ADR 0005: evaluation series must never have their content displayed. These are the content
# columns of the adjudication CSVs; `safe_table` refuses to show any of them.
CONTENT_COLUMNS = {
    "sentence_text",
    "evidence_text",
    "razon",
    "earlier_text",
    "later_text",
    "final_object",
    "final_content",
}
CONTENT_SUFFIXES = ("_object", "_content", "_notes", "_split", "_text")


def safe_table(records, columns):
    # Build a table from explicit label/id columns only; fail on any content column.
    banned = [c for c in columns if c in CONTENT_COLUMNS or c.endswith(CONTENT_SUFFIXES)]
    if banned:
        raise ValueError(f"Content columns are not displayable (ADR 0005): {banned}")
    return pd.DataFrame(records)[list(columns)]


# Guard: these files are committed, so a clone has them; fail clearly if any is missing.
def check_inputs(series):
    missing = [
        p.name
        for p in (
            PATHS.adjudicated(TaskKind.DECISIONS, series),
            PATHS.adjudicated(TaskKind.CANDIDATES, series),
            *annotator_csvs(PATHS.decisions_dir, series, "decisions"),
            *annotator_csvs(PATHS.relations_dir, series, "candidates"),
            PROJECT_ROOT / "reports" / "tables" / f"{series}.agreement.csv",
        )
        if not p.exists()
    ]
    if missing or len(annotator_csvs(PATHS.relations_dir, series, "candidates")) != 2:
        raise SystemExit(f"Faltan archivos de la Fase 2 para {series}: {missing}")


for _series in F2_SERIES:
    check_inputs(_series)
print(f"Series de la Fase 2 incluidas: {', '.join(F2_SERIES)}")

Series de la Fase 2 incluidas: ES2016, IS1003


---

# 1. Insumos, flujo y protección del contenido

## 1.1 De la máquina a la referencia adjudicada

Es el mismo flujo de la Fase 1 ([notebook 04](04-jss-anotacion-f1-calibracion.ipynb), §1.1),
con `<serie>` igual a `ES2016` o `IS1003`:

| Etapa | Quién | Archivo (en `data/processed/`) |
|---|---|---|
| Frases de decisión extraídas, con su evidencia | máquina | `decisions/<serie>.decisions.csv` |
| Pares candidatos de reuniones distintas | máquina (filtro de bloqueo) | `relations/<serie>.candidates.csv` |
| Tarea A, por separado | `gm` · `gv` | `decisions/<serie>.decisions.gm.csv` · `.gv.csv` |
| Tarea B, por separado | `gm` · `gv` | `relations/<serie>.candidates.gm.csv` · `.gv.csv` |
| Acuerdo entre anotadores | función `compute_series_agreement` | `reports/tables/<serie>.agreement.csv` |
| Adjudicación de los desacuerdos | `jss` | `decisions/<serie>.decisions.adjudicated.csv` · `relations/<serie>.candidates.adjudicated.csv` |

Las reglas de diseño no cambian: `gm` y `gv` no comparan etiquetas antes de la sesión
(independencia) y `jss` no etiqueta ninguna fila, solo resuelve donde difieren.

## 1.2 Qué cambia respecto de la Fase 1

En la Fase 1 las series (ES2015, TS3009) eran de desarrollo y se podían citar frases y razones.
ES2016 e IS1003 son de evaluación (ADR 0005): **el contenido no se muestra**. Lo que sí se puede mostrar
es la estructura del juicio: qué etiqueta puso cada persona, cuál fue el veredicto y a quién dio
la razón. Con eso alcanza para medir el acuerdo y describir el patrón de desacuerdos por pares
de etiquetas.

## 1.3 Verificación de la adjudicación y tamaño de cada etapa

In [2]:
# The adjudication is "complete" when the validator finds no issue in either CSV.
for _series in F2_SERIES:
    issues = validate_adjudication(_series, paths=PATHS)
    print(f"Problemas encontrados por validate-adjudication en {_series}: {len(issues)}")

# Size of each stage (row counts only), straight from the committed files.
sizes = pd.DataFrame(
    {
        _series: [
            len(pd.read_csv(PATHS.source(TaskKind.DECISIONS, _series))),
            len(pd.read_csv(PATHS.source(TaskKind.CANDIDATES, _series))),
            len(pd.read_csv(PATHS.adjudicated(TaskKind.DECISIONS, _series))),
            len(pd.read_csv(PATHS.adjudicated(TaskKind.CANDIDATES, _series))),
        ]
        for _series in F2_SERIES
    },
    index=[
        "decisions (máquina)",
        "candidates (máquina)",
        "decisions adjudicado (base + hijas)",
        "candidates adjudicado",
    ],
)
sizes.index.name = "filas por archivo"
display(sizes)

Problemas encontrados por validate-adjudication en ES2016: 0
Problemas encontrados por validate-adjudication en IS1003: 0


,ES2016,IS1003
filas por archivo,,
decisions (máquina),24,25
candidates (máquina),4,8
decisions adjudicado (base + hijas),26,25
candidates adjudicado,4,8


**Lectura.** En ES2016, las 24 decisiones base de la máquina pasan a 26 filas adjudicadas porque
una fila se declaró `compuesta` y se dividió en dos filas hijas (§2.3); los 4 pares quedan en 4.
En IS1003 las 25 decisiones base quedan en 25 filas (ninguna `compuesta`, §3.3) y los 8 pares
quedan en 8.

---

# 2. ES2016

## 2.1 Acuerdo en decisiones (Tarea A)

Misma medida que en la Fase 1: acuerdo crudo y kappa de Cohen sobre el `status` de las filas
base. Este kappa de estado **no** es uno de los tres de la Tarea D; es el diagnóstico adicional
descrito en el manual (§6) y **no se reporta**.

In [3]:
agreement = decision_agreement(SERIES, paths=PATHS)
a, b = agreement.annotator_a, agreement.annotator_b
print(f"Anotadores: {a} y {b}")
print(f"Filas base comparadas:           {agreement.n_items}")
print(
    f"Mismo estado:                    {agreement.n_status_agree} "
    f"({agreement.percent_agreement:.1%})"
)
print(f"Filas con acuerdo = si:          {agreement.n_acuerdo}")
print(f"Kappa de Cohen sobre el estado:  {agreement.status_kappa:.3f}  (DIAGNÓSTICO)")

Anotadores: gm y gv
Filas base comparadas:           24
Mismo estado:                    17 (70.8%)
Filas con acuerdo = si:          17
Kappa de Cohen sobre el estado:  -0.077  (DIAGNÓSTICO)


Cruce de lo que puso cada anotador (filas: `gm`; columnas: `gv`). La diagonal es el acuerdo.

In [4]:
contingency = (
    pd.DataFrame(
        status_contingency(SERIES, paths=PATHS), columns=[f"{a} (filas)", f"{b} (columnas)", "n"]
    )
    .pivot(index=f"{a} (filas)", columns=f"{b} (columnas)", values="n")
    .fillna(0)
    .astype(int)
)
display(contingency)

gv (columnas),compuesta,decision,no_decision
gm (filas),,,
decision,1,17,3
sin_soporte,0,3,0


**Lectura.**

- **17 coincidencias en 24 filas (70,8 %) y un kappa de estado de −0,077**: el acuerdo crudo
  es alto y el kappa es ligeramente negativo, es decir, no mejor que el azar. Parece una
  contradicción y no lo es (§2.2).
- **Las 17 coincidencias son todas `decision`.** No hay ninguna fila en que ambos pusieran
  `no_decision`, `sin_soporte` o `compuesta`. Los 7 desacuerdos son de dos formas: `gm` puso
  `decision` y `gv` algo más estricto (3 `no_decision` y 1 `compuesta`), o `gm` puso
  `sin_soporte` y `gv` `decision` (3 filas). Es la primera vez que aparece este segundo sentido
  del desacuerdo: en la Fase 1 siempre era `gm` quien aceptaba más.

## 2.2 Por qué el kappa es −0,077 con un 71 % de acuerdo

El kappa vale `(p_o − p_e) / (1 − p_e)`: `p_o` es el acuerdo observado y `p_e` el que se
esperaría por azar dado el uso que cada persona hizo de cada etiqueta. Aquí ambas personas
dijeron `decision` casi siempre, y son las únicas que coinciden; las etiquetas minoritarias que
usó cada una (`sin_soporte` en `gm`; `no_decision` y `compuesta` en `gv`) **no son las mismas**,
así que nunca coinciden. El azar "explica" entonces más que lo observado y el kappa cae bajo cero.
Es la paradoja de la prevalencia ya vista en la Fase 1 ([notebook 04](04-jss-anotacion-f1-calibracion.ipynb), §6.3).

In [5]:
distribution = annotator_status_distribution(SERIES, paths=PATHS)
usage = pd.DataFrame({ini: distribution[ini] for ini in (a, b)}).fillna(0).astype(int)
usage.loc["total"] = usage.sum()
display(usage)

n_items = agreement.n_items
p_o = agreement.percent_agreement
p_e = chance_agreement(SERIES, paths=PATHS)
print(f"Acuerdo observado p_o = {agreement.n_status_agree}/{n_items} = {p_o:.3f}")
print(f"Acuerdo esperado  p_e = {p_e:.3f}")
print(
    f"kappa = (p_o - p_e) / (1 - p_e) = ({p_o:.3f} - {p_e:.3f}) / (1 - {p_e:.3f}) "
    f"= {agreement.status_kappa:.3f}"
)

,gm,gv
decision,21,20
sin_soporte,3,0
compuesta,0,1
no_decision,0,3
total,24,24


Acuerdo observado p_o = 17/24 = 0.708
Acuerdo esperado  p_e = 0.729
kappa = (p_o - p_e) / (1 - p_e) = (0.708 - 0.729) / (1 - 0.729) = -0.077


**Lectura.** `p_e` (0,729) supera a `p_o` (0,708) porque ambos concentran su uso en `decision`
(21 y 20 de 24 filas) y no comparten ninguna etiqueta minoritaria. Con esta forma de datos el
kappa no tiene de dónde sacar información: no dice que las personas entendieran la guía peor que
por azar, dice que el acuerdo que hubo está casi todo en la categoría que ambas usan
casi siempre. Por eso el diagnóstico útil es **qué difirió** (§2.4), no el valor del kappa.

## 2.3 Referencia adjudicada: estados finales

`jss` adjudicó las 7 filas en que difirieron. Donde coincidieron, el CSV copia la etiqueta
común a las columnas `final_*`. La referencia distingue las filas base de las filas hijas que
nacen de una `compuesta`.

In [6]:
counts = final_status_counts(SERIES, paths=PATHS)
status_table = (
    pd.DataFrame({"filas base": counts.base, "filas hijas (de compuesta)": counts.children})
    .fillna(0)
    .astype(int)
)
status_table.loc["total"] = status_table.sum()
display(status_table)

# The compuesta row and its children: identifiers and final label only (no text columns).
children = safe_table(
    compuesta_children(SERIES, paths=PATHS), ["parent_id", "decision_id", "final_status"]
)
display(children)

,filas base,filas hijas (de compuesta)
compuesta,1,0
decision,19,2
no_decision,3,0
sin_soporte,1,0
total,24,2


,parent_id,decision_id,final_status
0,ES2016c.d03,ES2016c.d03-1,decision
1,ES2016c.d03,ES2016c.d03-2,decision


**Lectura.**

- **De las 24 decisiones base, 19 quedan como `decision`, 3 como `no_decision`, 1 como
  `sin_soporte` y 1 como `compuesta`.** La `compuesta` no es un veredicto final sino una
  instrucción: se divide en filas hijas.
- **Las 2 filas hijas (`ES2016c.d03-1` y `ES2016c.d03-2`) son `decision`.** La referencia de
  ES2016 queda con **21 filas `decision`** (19 base + 2 hijas), 3 `no_decision` y 1 `sin_soporte`.
- Los `no_decision` se conservan a propósito: miden los falsos positivos de la extracción
  (manual §2).

## 2.4 Los desacuerdos de decisiones: a quién dio la razón el veredicto

Cada fila de la tabla es un desacuerdo. `sided_with` indica el anotador cuya etiqueta coincide
con el veredicto final. **No se muestran las razones**: son texto sobre contenido de evaluación
(ADR 0005).

In [7]:
disagreements = safe_table(
    resolved_disagreements(SERIES, paths=PATHS),
    ["decision_id", f"{a}_status", f"{b}_status", "final_status", "sided_with"],
)
display(disagreements)
print("Veredicto final coincide con:", side_taken_counts(SERIES, paths=PATHS))

,decision_id,gm_status,gv_status,final_status,sided_with
0,ES2016b.d04,decision,no_decision,no_decision,gv
1,ES2016b.d05,decision,no_decision,no_decision,gv
2,ES2016b.d06,decision,no_decision,no_decision,gv
3,ES2016b.d09,sin_soporte,decision,decision,gv
4,ES2016c.d03,decision,compuesta,compuesta,gv
5,ES2016d.d06,sin_soporte,decision,sin_soporte,gm
6,ES2016d.d07,sin_soporte,decision,decision,gv


Veredicto final coincide con: {'gm': 1, 'gv': 6}


**Lectura.**

- **El veredicto coincidió con `gv` en 6 de los 7 desacuerdos y con `gm` en 1.** En la Fase 1
  era al revés (con `gm` en 13 de 16 en ES2015). El resultado depende de qué frases tenía cada
  serie, y **no se interpreta como que un anotador anotó mejor**: mide la distancia entre dos
  lecturas y la de la tercera persona.
- Los desacuerdos se agrupan por pares de etiquetas, y cada grupo corresponde a un criterio del
  manual. Se presentan a continuación solo con etiquetas, conteos e identificadores.

### Patrón de los desacuerdos por pares de etiquetas

La columna "criterio del manual" asigna a cada grupo el criterio del manual que le
corresponde según el par de etiquetas y el veredicto; es una categoría general y no cita ningún
contenido.

In [8]:
CRITERIO = {
    (
        "decision",
        "no_decision",
        "no_decision",
    ): "umbral: meta o aspiración, no elección de diseño (manual §2(a))",
    (
        "sin_soporte",
        "decision",
        "decision",
    ): "la transcripción de la reunión sí contiene la frase (manual §2(b))",
    (
        "sin_soporte",
        "decision",
        "sin_soporte",
    ): "la transcripción contradice la frase; el caso se reinterpretó (manual §2(b))",
    (
        "decision",
        "compuesta",
        "compuesta",
    ): "frase con decisiones separables que no se dividió (regla (a) del notebook 04, §5)",
}

dis = disagreements.rename(columns={f"{a}_status": a, f"{b}_status": b})
pattern = (
    dis.groupby([a, b, "final_status"])
    .agg(n=("decision_id", "size"), decision_ids=("decision_id", ", ".join))
    .reset_index()
)
pattern["criterio del manual"] = [
    CRITERIO.get((r[a], r[b], r["final_status"]), "sin clasificar") for _, r in pattern.iterrows()
]
display(pattern)
assert (pattern["criterio del manual"] != "sin clasificar").all()
print(f"Total de desacuerdos: {pattern['n'].sum()}")

,gm,gv,final_status,n,decision_ids,criterio del manual
0,decision,compuesta,compuesta,1,ES2016c.d03,"frase con decisiones separables que no se dividió (regla (a) del notebook 04, §5)"
1,decision,no_decision,no_decision,3,"ES2016b.d04, ES2016b.d05, ES2016b.d06","umbral: meta o aspiración, no elección de diseño (manual §2(a))"
2,sin_soporte,decision,decision,2,"ES2016b.d09, ES2016d.d07",la transcripción de la reunión sí contiene la frase (manual §2(b))
3,sin_soporte,decision,sin_soporte,1,ES2016d.d06,la transcripción contradice la frase; el caso se reinterpretó (manual §2(b))


Total de desacuerdos: 7


**Lectura.**

| Patrón | Casos | Qué muestra |
|---|---|---|
| `decision` / `no_decision` → `no_decision` | 3 | Umbral entre decisión y meta o aspiración (manual §2(a)). Es el mismo vacío de la Fase 1, ahora con `gv` más estricto y el veredicto de su lado |
| `sin_soporte` / `decision` → `decision` | 2 | `gm` marcó falta de soporte donde la transcripción de la reunión sí contiene la frase (manual §2(b)): la evidencia enlazada no basta, pero la reunión sí lo respalda |
| `sin_soporte` / `decision` → `sin_soporte` | 1 | Único caso en que el veredicto fue de `gm`: la transcripción contradice la frase y fue reinterpretada |
| `decision` / `compuesta` → `compuesta` | 1 | Frase con decisiones separables que `gm` no dividió |

- **4 de los 7 desacuerdos son de umbral o de separabilidad, y 3 son de soporte.** La corrección
  de la guía hecha tras la Fase 1 no eliminó el desacuerdo de umbral: lo redujo (de 14 de 29
  filas en ES2015 a 3 de 24 aquí), pero sigue siendo el patrón más frecuente.
- **Aparece un desacuerdo nuevo respecto de la Fase 1: `sin_soporte` frente a `decision` con
  `gm` como quien duda.** Dos de los tres casos se resolvieron a favor de `decision` porque la
  guía manda leer la reunión y no solo `evidence_text` (manual §2, regla de oro).

## 2.5 Acuerdo en pares (Tarea B)

El filtro de bloqueo dejó **4 pares** en ES2016. Con tan pocos pares los kappas de la Tarea D
(existencia, tipo, dirección) son inestables o indefinidos. Se calculan con la misma función que
el comando `afg gold agreement` y se contrastan con la tabla versionada.

In [9]:
pairs = safe_table(
    adjudicated_pairs(SERIES, paths=PATHS),
    [
        "pair_id",
        "earlier_decision_id",
        "later_decision_id",
        f"{a}_relation",
        f"{b}_relation",
        "acuerdo",
        "final_relation",
    ],
)
display(pairs)

result = compute_series_agreement(annotator_csvs(PATHS.relations_dir, SERIES, "candidates"))
kappa_table = pd.DataFrame(
    [
        ("existencia", result.n_items, result.existence_kappa, result.existence_disagreements),
        ("tipo", result.n_type_items, result.type_kappa, result.type_disagreements),
        ("dirección", result.n_items, result.direction_kappa, result.direction_disagreements),
    ],
    columns=["eje", "n", "kappa", "pares en desacuerdo"],
)
kappa_table["pares en desacuerdo"] = kappa_table["pares en desacuerdo"].map(", ".join)
display(kappa_table)

# Cross-check against the table already committed by `afg gold agreement`.
committed = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / f"{SERIES}.agreement.csv")
display(committed[["axis", "n_items", "kappa", "disagreeing_pair_ids"]])

profiles = pd.DataFrame(
    [
        {
            "eje": p.axis,
            "n": p.n_items,
            "acuerdo crudo": p.percent_agreement,
            "categorías usadas (gm)": ", ".join(p.labels_a),
            "categorías usadas (gv)": ", ".join(p.labels_b),
            "kappa definido": p.kappa_defined,
        }
        for p in series_axis_profiles(annotator_csvs(PATHS.relations_dir, SERIES, "candidates"))
    ]
)
display(profiles)

,pair_id,earlier_decision_id,later_decision_id,gm_relation,gv_relation,acuerdo,final_relation
0,ES2016.p001,ES2016b.d11,ES2016c.d03,refina,refina,si,refina
1,ES2016.p002,ES2016c.d03,ES2016d.d03,reafirma,reafirma,si,reafirma
2,ES2016.p003,ES2016c.d02,ES2016d.d05,reafirma,reafirma,si,reafirma
3,ES2016.p004,ES2016c.d01,ES2016d.d02,refina,refina,si,refina


,eje,n,kappa,pares en desacuerdo
0,existencia,4,NaN,
1,tipo,4,1.0,
2,dirección,4,NaN,


,axis,n_items,kappa,disagreeing_pair_ids
0,existence,4,NaN,NaN
1,type,4,1.0,NaN
2,direction,4,NaN,NaN


,eje,n,acuerdo crudo,categorías usadas (gm),categorías usadas (gv),kappa definido
0,existence,4,1.0,exists,exists,False
1,type,4,1.0,"reafirma, refina","reafirma, refina",True
2,direction,4,1.0,si,si,False


**Lectura.**

- **Acuerdo total en los 4 pares.** Los dos anotadores coincidieron en la relación de cada par
  y no hubo desacuerdos que adjudicar en la Tarea B.
- **Tipo: kappa 1,0 sobre 4 pares.** Ambos usaron más de una etiqueta (`refina` y `reafirma`)
  y coincidieron siempre, de modo que el kappa está definido y vale 1. Con 4 observaciones, un
  solo par distinto lo habría movido mucho: el número es exacto pero poco informativo.
- **Existencia y dirección: kappa indefinido.** Ambos anotadores vieron un vínculo en los 4
  pares y los 4 con la dirección correcta: una sola categoría en cada eje. El acuerdo esperado
  vale 1, el denominador de `(p_o − p_e) / (1 − p_e)` se anula y el cociente es 0/0
  ([notebook 04](04-jss-anotacion-f1-calibracion.ipynb), §3.4). Mostrar "1,0" o "0,0" sería
  inventar un valor; se reporta como indefinido junto con el acuerdo crudo (100 %).

## 2.6 ES2016 frente a la Fase 1

La misma función (`decision_agreement`) sobre las tres series. ES2015 y TS3009 son de la Fase 1
(diagnóstico, series de desarrollo); ES2016 es de la Fase 2.

In [10]:
rows = []
for fase, series_list in (("Fase 1", F1_SERIES), ("Fase 2", [SERIES])):
    for series in series_list:
        res = decision_agreement(series, paths=PATHS)
        rows.append(
            {
                "fase": fase,
                "serie": series,
                "filas base": res.n_items,
                "mismo estado": res.n_status_agree,
                "acuerdo crudo": res.percent_agreement,
                "acuerdo esperado por azar": chance_agreement(series, paths=PATHS),
                "kappa de estado": res.status_kappa,
            }
        )
display(
    pd.DataFrame(rows)
    .style.format(
        {
            "acuerdo crudo": "{:.1%}",
            "acuerdo esperado por azar": "{:.3f}",
            "kappa de estado": "{:.3f}",
        }
    )
    .hide(axis="index")
)

fase,serie,filas base,mismo estado,acuerdo crudo,acuerdo esperado por azar,kappa de estado
Fase 1,ES2015,29,13,44.8%,0.391,0.094
Fase 1,TS3009,24,20,83.3%,0.833,0.000
Fase 2,ES2016,24,17,70.8%,0.729,-0.077


**Lectura.**

- **Acuerdo crudo: 45 % (ES2015), 83 % (TS3009), 71 % (ES2016).** ES2016 queda entre las dos
  series de la Fase 1: mucho mejor que antes de corregir la guía, algo peor que TS3009.
- **Kappa de estado: 0,094, 0,000 y −0,077.** Ninguno es interpretable por separado: en las tres
  series al menos una persona concentra casi todo en `decision`, así que el azar explica una
  parte grande del acuerdo (paradoja de la prevalencia, §2.2). El kappa de estado no mide aquí
  si la guía mejoró; el conteo de desacuerdos por patrón (§2.4) sí.
- **Todos los kappas de esta tabla son diagnósticos y no se reportan en la tesis.** Los
  reportables son los de la Tarea D (§2.5 y §5).
- **Comparar series de tamaño y composición distintas tiene un límite:** 24 a 29 filas por
  serie no sostienen diferencias finas entre 71 % y 83 %.



## 2.7 Qué implica para las series siguientes

- **Los 7 desacuerdos de ES2016 caen en el manual §2(a) y §2(b)**: umbral entre `decision` y
  `no_decision`, cuándo la reunión sí respalda una frase que la evidencia enlazada no cubre, y
  separabilidad de una `compuesta`. La revisión conjunta de esos criterios no puede mirar casos
  de ES2016 (ADR 0005); se hace sobre los criterios y los casos de desarrollo del manual. El
  resultado de IS1003 sobre ese mismo criterio está en §3.7.
- **La Tarea C de ES2016 (issue #155) se juzga contra la referencia adjudicada** (PR #157), no
  contra las etiquetas de un anotador: la referencia es la que resultó de este notebook.
- **Los kappas de pares de las próximas series también tendrán pocos pares** (IS1003, TS3005 y
  ES2008), y se leerán con el mismo límite de §2.5.

---

# 3. IS1003

La misma estructura de §2, aplicada a IS1003 (`SERIES_IS`). El procedimiento de lectura no
cambia: solo etiquetas, conteos e identificadores, nunca contenido (ADR 0005).

## 3.1 Acuerdo en decisiones (Tarea A)

Misma medida que en §2.1: acuerdo crudo y kappa de Cohen sobre el `status` de las filas base. El
kappa de estado es diagnóstico y **no se reporta**.

In [11]:
is_agreement = decision_agreement(SERIES_IS, paths=PATHS)
print(f"Anotadores: {is_agreement.annotator_a} y {is_agreement.annotator_b}")
print(f"Filas base comparadas:           {is_agreement.n_items}")
print(
    f"Mismo estado:                    {is_agreement.n_status_agree} "
    f"({is_agreement.percent_agreement:.1%})"
)
print(f"Filas con acuerdo = si:          {is_agreement.n_acuerdo}")
print(f"Kappa de Cohen sobre el estado:  {is_agreement.status_kappa:.3f}  (DIAGNÓSTICO)")

Anotadores: gm y gv
Filas base comparadas:           25
Mismo estado:                    22 (88.0%)
Filas con acuerdo = si:          22
Kappa de Cohen sobre el estado:  0.000  (DIAGNÓSTICO)


Cruce de lo que puso cada anotador (filas: `gm`; columnas: `gv`). La diagonal es el acuerdo.

In [12]:
is_contingency = (
    pd.DataFrame(
        status_contingency(SERIES_IS, paths=PATHS), columns=[f"{a} (filas)", f"{b} (columnas)", "n"]
    )
    .pivot(index=f"{a} (filas)", columns=f"{b} (columnas)", values="n")
    .fillna(0)
    .astype(int)
)
display(is_contingency)

gv (columnas),decision
gm (filas),
decision,22
descartar,1
no_decision,2


**Lectura.**

- **22 coincidencias en 25 filas (88,0 %) y un kappa de estado de 0,000**: el acuerdo crudo es
  el más alto de las cuatro series con diagnóstico y el kappa es cero. Es la misma aparente
  contradicción de ES2016, más marcada (§3.2).
- **Las 22 coincidencias son todas `decision`.** Los 3 desacuerdos tienen la misma forma: `gv`
  puso `decision` en las 25 filas y `gm` puso algo más estricto en 3 (2 `no_decision` y 1
  `descartar`). No hay `sin_soporte` ni `compuesta` en ninguno de los dos anotadores. Es el
  sentido contrario al de la Fase 1: aquí quien acepta más es siempre `gv`.

## 3.2 Por qué el kappa es 0,00 con un 88 % de acuerdo

Es el mismo mecanismo de §2.2, en su caso extremo: `gv` usó una sola etiqueta, `decision`, en
todas las filas. Cuando una de las dos personas usa una única categoría, el acuerdo esperado por
azar es igual a la proporción con que la otra persona usó esa categoría, y el kappa solo puede
valer cero o menos, sea cual sea el acuerdo observado.

In [13]:
is_distribution = annotator_status_distribution(SERIES_IS, paths=PATHS)
is_usage = pd.DataFrame({ini: is_distribution[ini] for ini in (a, b)}).fillna(0).astype(int)
is_usage.loc["total"] = is_usage.sum()
display(is_usage)

is_p_o = is_agreement.percent_agreement
is_p_e = chance_agreement(SERIES_IS, paths=PATHS)
print(
    f"Acuerdo observado p_o = {is_agreement.n_status_agree}/{is_agreement.n_items} = {is_p_o:.3f}"
)
print(f"Acuerdo esperado  p_e = {is_p_e:.3f}")
print(
    f"kappa = (p_o - p_e) / (1 - p_e) = ({is_p_o:.3f} - {is_p_e:.3f}) / (1 - {is_p_e:.3f}) "
    f"= {is_agreement.status_kappa:.3f}"
)

,gm,gv
decision,22,25
descartar,1,0
no_decision,2,0
total,25,25


Acuerdo observado p_o = 22/25 = 0.880
Acuerdo esperado  p_e = 0.880
kappa = (p_o - p_e) / (1 - p_e) = (0.880 - 0.880) / (1 - 0.880) = 0.000


**Lectura.** `p_e` (0,880) es igual a `p_o` (0,880): `gv` marcó `decision` en 25 de 25 filas y
`gm` en 22 de 25, de modo que el azar "explica" exactamente el acuerdo observado y el kappa
vale cero. No dice que las personas coincidieran solo por azar: dice que con una categoría
usada sin excepción por una de ellas, el kappa no tiene información que aportar. El diagnóstico
útil es otra vez **qué difirió** (§3.4), no el valor del kappa.

## 3.3 Referencia adjudicada: estados finales

`jss` adjudicó las 3 filas en que difirieron. Donde coincidieron, el CSV copia la etiqueta común
a las columnas `final_*`. IS1003 no tiene filas `compuesta`, por lo que no hay filas hijas.

In [14]:
is_counts = final_status_counts(SERIES_IS, paths=PATHS)
is_status_table = (
    pd.DataFrame({"filas base": is_counts.base, "filas hijas (de compuesta)": is_counts.children})
    .fillna(0)
    .astype(int)
)
is_status_table.loc["total"] = is_status_table.sum()
display(is_status_table)

# IS1003 has no compuesta row: the children table must be empty.
is_children = compuesta_children(SERIES_IS, paths=PATHS)
print(f"Filas hijas de compuesta en {SERIES_IS}: {len(is_children)}")

,filas base,filas hijas (de compuesta)
decision,22,0
descartar,1,0
no_decision,2,0
total,25,0


Filas hijas de compuesta en IS1003: 0


**Lectura.**

- **De las 25 decisiones base, 22 quedan como `decision`, 2 como `no_decision` y 1 como
  `descartar`.** La referencia de IS1003 queda con **22 filas `decision`**.
- `descartar` retira la fila de la referencia: no es un falso positivo de la extracción sino una
  duplicada dentro de la misma reunión (manual §2). Los 2 `no_decision` se conservan a propósito:
  miden los falsos positivos de la extracción.

## 3.4 Los desacuerdos de decisiones: a quién dio la razón el veredicto

Cada fila de la tabla es un desacuerdo. `sided_with` indica el anotador cuya etiqueta coincide
con el veredicto final. **No se muestran las razones** (ADR 0005).

In [15]:
is_disagreements = safe_table(
    resolved_disagreements(SERIES_IS, paths=PATHS),
    ["decision_id", f"{a}_status", f"{b}_status", "final_status", "sided_with"],
)
display(is_disagreements)
print("Veredicto final coincide con:", side_taken_counts(SERIES_IS, paths=PATHS))

,decision_id,gm_status,gv_status,final_status,sided_with
0,IS1003a.d03,no_decision,decision,no_decision,gm
1,IS1003b.d01,descartar,decision,descartar,gm
2,IS1003c.d02,no_decision,decision,no_decision,gm


Veredicto final coincide con: {'gm': 3}


**Lectura.**

- **El veredicto coincidió con `gm` en los 3 desacuerdos.** En ES2016 fue lo contrario (con `gv`
  en 6 de 7). Con 3 casos esto no se interpreta como que un anotador anotó mejor: describe qué
  tipo de frase tenía cada serie y quién fue más estricto en ella.

### Patrón de los desacuerdos por pares de etiquetas

Se reutiliza el diccionario `CRITERIO` de §2.4 y se agregan los dos pares de etiquetas nuevos
(`no_decision`/`decision` y `descartar`/`decision`). Cada criterio es una categoría general y no
cita contenido.

In [16]:
CRITERIO_IS = {
    **CRITERIO,
    (
        "no_decision",
        "decision",
        "no_decision",
    ): "umbral: meta o aspiración, no elección de diseño (manual §2(a))",
    (
        "descartar",
        "decision",
        "descartar",
    ): "decisión repetida en la misma reunión: se conserva la fila explícita (manual §2)",
}

is_dis = is_disagreements.rename(columns={f"{a}_status": a, f"{b}_status": b})
is_pattern = (
    is_dis.groupby([a, b, "final_status"])
    .agg(n=("decision_id", "size"), decision_ids=("decision_id", ", ".join))
    .reset_index()
)
is_pattern["criterio del manual"] = [
    CRITERIO_IS.get((r[a], r[b], r["final_status"]), "sin clasificar")
    for _, r in is_pattern.iterrows()
]
display(is_pattern)
assert (is_pattern["criterio del manual"] != "sin clasificar").all()
print(f"Total de desacuerdos: {is_pattern['n'].sum()}")

,gm,gv,final_status,n,decision_ids,criterio del manual
0,descartar,decision,descartar,1,IS1003b.d01,decisión repetida en la misma reunión: se conserva la fila explícita (manual §2)
1,no_decision,decision,no_decision,2,"IS1003a.d03, IS1003c.d02","umbral: meta o aspiración, no elección de diseño (manual §2(a))"


Total de desacuerdos: 3


**Lectura.**

| Patrón | Casos | Qué muestra |
|---|---|---|
| `no_decision` / `decision` → `no_decision` | 2 | Umbral entre decisión y meta o aspiración (manual §2(a)): `gv` aceptó como decisión una frase que expresa un objetivo o un requisito sin que el equipo elija nada concreto |
| `descartar` / `decision` → `descartar` | 1 | Decisión repetida dentro de la misma reunión: `gm` la marcó como duplicada de otra fila y `jss` conservó la fila que la enuncia de forma explícita |

- **2 de los 3 desacuerdos repiten el patrón de umbral de ES2016** (`ES2016b.d04` a `d06` allí): meta
  o aspiración marcada como decisión. Es el vacío de la Fase 1 y sigue sin cerrarse, aunque en
  IS1003 pesa menos (2 de 25 filas).
- **El tercero es de otra naturaleza:** no discute si hay decisión, sino si la fila cuenta una
  vez o dos. La regla del manual existe; en este caso `gv` no la aplicó.

## 3.5 Acuerdo en pares (Tarea B)

El filtro de bloqueo dejó **8 pares** en IS1003, el doble que en ES2016. Se calculan los kappas
con la misma función que el comando `afg gold agreement` y se contrastan con la tabla versionada.

In [17]:
is_pairs = safe_table(
    adjudicated_pairs(SERIES_IS, paths=PATHS),
    [
        "pair_id",
        "earlier_decision_id",
        "later_decision_id",
        f"{a}_relation",
        f"{b}_relation",
        "acuerdo",
        "final_relation",
    ],
)
display(is_pairs)

is_csvs = annotator_csvs(PATHS.relations_dir, SERIES_IS, "candidates")
is_result = compute_series_agreement(is_csvs)
is_kappa_table = pd.DataFrame(
    [
        (
            "existencia",
            is_result.n_items,
            is_result.existence_kappa,
            is_result.existence_disagreements,
        ),
        ("tipo", is_result.n_type_items, is_result.type_kappa, is_result.type_disagreements),
        (
            "dirección",
            is_result.n_items,
            is_result.direction_kappa,
            is_result.direction_disagreements,
        ),
    ],
    columns=["eje", "n", "kappa", "pares en desacuerdo"],
)
is_kappa_table["pares en desacuerdo"] = is_kappa_table["pares en desacuerdo"].map(", ".join)
display(is_kappa_table)

# Cross-check against the table already committed by `afg gold agreement`.
is_committed = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / f"{SERIES_IS}.agreement.csv")
display(is_committed[["axis", "n_items", "kappa", "disagreeing_pair_ids"]])

is_profiles = pd.DataFrame(
    [
        {
            "eje": p.axis,
            "n": p.n_items,
            "acuerdo crudo": p.percent_agreement,
            "categorías usadas (gm)": ", ".join(p.labels_a),
            "categorías usadas (gv)": ", ".join(p.labels_b),
            "kappa definido": p.kappa_defined,
        }
        for p in series_axis_profiles(is_csvs)
    ]
)
display(is_profiles)

,pair_id,earlier_decision_id,later_decision_id,gm_relation,gv_relation,acuerdo,final_relation
0,IS1003.p001,IS1003a.d02,IS1003b.d02,no_relacionada,no_relacionada,si,no_relacionada
1,IS1003.p002,IS1003b.d02,IS1003c.d07,no_relacionada,no_relacionada,si,no_relacionada
2,IS1003.p003,IS1003b.d05,IS1003d.d07,refina,reafirma,no,refina
3,IS1003.p004,IS1003b.d04,IS1003d.d06,no_relacionada,no_relacionada,si,no_relacionada
4,IS1003.p005,IS1003b.d06,IS1003d.d07,no_relacionada,no_relacionada,si,no_relacionada
5,IS1003.p006,IS1003c.d04,IS1003d.d04,refina,refina,si,refina
6,IS1003.p007,IS1003c.d06,IS1003d.d02,reemplaza,reemplaza,si,reemplaza
7,IS1003.p008,IS1003c.d06,IS1003d.d03,no_relacionada,no_relacionada,si,no_relacionada


,eje,n,kappa,pares en desacuerdo
0,existencia,8,1.0,
1,tipo,3,0.5,IS1003.p003
2,dirección,8,NaN,


,axis,n_items,kappa,disagreeing_pair_ids
0,existence,8,1.0,NaN
1,type,3,0.5,IS1003.p003
2,direction,8,NaN,NaN


,eje,n,acuerdo crudo,categorías usadas (gm),categorías usadas (gv),kappa definido
0,existence,8,1.000000,"exists, no_relation","exists, no_relation",True
1,type,3,0.666667,"reemplaza, refina","reafirma, reemplaza, refina",True
2,direction,8,1.000000,si,si,False


**Lectura.**

- **Existencia: kappa 1,000 sobre 8 pares.** Ambos anotadores vieron vínculo en 3 pares y
  ausencia en 5, y coincidieron en todos. A diferencia de ES2016, aquí el kappa de existencia
  está definido porque ambos usaron las dos categorías.
- **Tipo: kappa 0,500 sobre 3 pares**, los que ambos consideraron vinculados. Hubo un
  desacuerdo, `IS1003.p003`: `gm` puso `refina` y `gv` `reafirma`; el veredicto fue `refina`
  (con `gm`). Con 3 observaciones un solo par distinto mueve el kappa de 1,0 a 0,5, por lo que
  el número es exacto pero poco informativo. El desacuerdo es de fondo y no de etiquetado: la
  frontera entre precisar el contenido (`refina`) y repetirlo (`reafirma`).
- **Dirección: kappa indefinido.** En los 8 pares ambos marcaron la dirección como correcta:
  una sola categoría. El acuerdo esperado vale 1, el cociente es 0/0 y se reporta como
  indefinido junto con el acuerdo crudo (100 %), no como 1,0 ni como 0,0.
- **La adjudicación de pares se reduce a 1 par** (`IS1003.p003`) de 8.

## 3.6 IS1003 frente a ES2016 y la Fase 1

La misma función (`decision_agreement`) sobre las cuatro series: ES2015 y TS3009 de la Fase 1
(series de desarrollo) y ES2016 e IS1003 de la Fase 2.

In [18]:
rows = []
for fase, series_list in (("Fase 1", F1_SERIES), ("Fase 2", F2_SERIES)):
    for series in series_list:
        res = decision_agreement(series, paths=PATHS)
        rows.append(
            {
                "fase": fase,
                "serie": series,
                "filas base": res.n_items,
                "mismo estado": res.n_status_agree,
                "acuerdo crudo": res.percent_agreement,
                "acuerdo esperado por azar": chance_agreement(series, paths=PATHS),
                "kappa de estado": res.status_kappa,
            }
        )
display(
    pd.DataFrame(rows)
    .style.format(
        {
            "acuerdo crudo": "{:.1%}",
            "acuerdo esperado por azar": "{:.3f}",
            "kappa de estado": "{:.3f}",
        }
    )
    .hide(axis="index")
)

fase,serie,filas base,mismo estado,acuerdo crudo,acuerdo esperado por azar,kappa de estado
Fase 1,ES2015,29,13,44.8%,0.391,0.094
Fase 1,TS3009,24,20,83.3%,0.833,0.000
Fase 2,ES2016,24,17,70.8%,0.729,-0.077
Fase 2,IS1003,25,22,88.0%,0.880,0.000


**Lectura.**

- **Acuerdo crudo: 45 % (ES2015), 83 % (TS3009), 71 % (ES2016), 88 % (IS1003).** IS1003 es el
  más alto de las cuatro series, pero la tabla no permite atribuirlo a la guía: son series
  distintas, de 24 a 29 filas, y la diferencia entre 88 % y 83 % es de pocas filas.
- **Kappa de estado: 0,094, 0,000, −0,077 y 0,000.** Siguen siendo no interpretables por
  separado (paradoja de la prevalencia, §2.2 y §3.2) y diagnósticos: no se reportan en la tesis.
- **El conteo de desacuerdos por patrón sí es comparable:** 3 de 25 filas en IS1003 (12 %),
  frente a 7 de 24 en ES2016 (29 %) y 14 de 29 por umbral en ES2015. Los desacuerdos de umbral
  bajaron a 2 de 25; no desaparecieron.

## 3.7 Qué implica para las series siguientes

- **No hace falta corregir la guía.** Los 3 desacuerdos caen en criterios que el manual ya
  tiene: el umbral de §2(a) (2 casos) y la regla de decisiones repetidas en la misma reunión
  (1 caso). Ninguno revela un vacío nuevo.
- **Antes de TS3005, `jss`, `gv` y `gm` deben repasar juntos el manual §2(a) y la regla de
  duplicados.** El umbral entre decisión y meta o aspiración reaparece en la segunda serie de
  evaluación, ahora con `gv` como quien acepta más, y el duplicado dentro de la
  reunión no lo aplicó uno de los dos anotadores. El repaso se hace sobre los criterios y
  los casos de desarrollo del manual, sin mirar casos de IS1003 (ADR 0005).
- **La Tarea C de IS1003 se juzga contra la referencia adjudicada** y no contra las etiquetas de
  un anotador.
- **El kappa de tipo de IS1003 (0,500 sobre 3 pares) se lee con el límite de §3.5**: un par
  decide entre 1,0 y 0,5. TS3005 y ES2008 tendrán pocos pares también.

---

# 4. Series pendientes

Cada serie se agrega como una sección con la misma estructura que §2 y §3 (acuerdo en decisiones,
estados finales, desacuerdos por pares de etiquetas, pares y comparación). El procedimiento al
cerrar una serie es: (1) añadirla a `F2_SERIES` en la primera celda, (2) copiar las celdas de
§3 con su propia variable de serie, (3) actualizar la tabla de §5.

| Sección | Serie | Estado |
|---|---|---|
| 4.1 | TS3005 | pendiente |
| 4.2 | ES2008 | pendiente |

---

# 5. Resumen entre series de control

In [19]:
summary = []
for series in F2_SERIES:
    res_a = decision_agreement(series, paths=PATHS)
    res_b = compute_series_agreement(annotator_csvs(PATHS.relations_dir, series, "candidates"))
    summary.append(
        {
            "serie": series,
            "decisiones base": res_a.n_items,
            "acuerdo crudo (A)": res_a.percent_agreement,
            "kappa de estado (diagnóstico)": res_a.status_kappa,
            "pares": res_b.n_items,
            "kappa existencia": res_b.existence_kappa,
            "kappa tipo": res_b.type_kappa,
            "kappa dirección": res_b.direction_kappa,
            "veredicto con gm": side_taken_counts(series, paths=PATHS).get("gm", 0),
            "veredicto con gv": side_taken_counts(series, paths=PATHS).get("gv", 0),
        }
    )
display(
    pd.DataFrame(summary)
    .style.format(
        {
            "acuerdo crudo (A)": "{:.1%}",
            "kappa de estado (diagnóstico)": "{:.3f}",
            "kappa existencia": "{:.3f}",
            "kappa tipo": "{:.3f}",
            "kappa dirección": "{:.3f}",
        },
        na_rep="indefinido",
    )
    .hide(axis="index")
)
print(f"Series de control adjudicadas: {len(F2_SERIES)} de 4")

serie,decisiones base,acuerdo crudo (A),kappa de estado (diagnóstico),pares,kappa existencia,kappa tipo,kappa dirección,veredicto con gm,veredicto con gv
ES2016,24,70.8%,-0.077,4,indefinido,1.000,indefinido,1,6
IS1003,25,88.0%,0.000,8,1.000,0.500,indefinido,3,0


Series de control adjudicadas: 2 de 4


**Lectura.** Con dos series adjudicadas la tabla ya permite una primera comparación, pero con las
cautelas de §2.5 y §3.5. Las columnas de la Tarea D (existencia, tipo, dirección) son las
reportables; "kappa de estado" es diagnóstico.

- **Decisiones.** El acuerdo crudo sube de 71 % (ES2016) a 88 % (IS1003) y los dos kappas de
  estado son cero o negativos por la misma razón: un anotador concentra casi todo en `decision`.
- **Veredictos.** En ES2016 el veredicto coincidió con `gv` en 6 de 7 desacuerdos; en IS1003,
  con `gm` en 3 de 3. No hay un anotador sistemáticamente más cercano a la adjudicación.
- **Pares.** Existencia está definida solo en IS1003 (1,000 sobre 8 pares); tipo vale 1,000
  sobre 4 pares en ES2016 y 0,500 sobre 3 en IS1003; dirección es indefinida en ambas.
  Agrupar esos valores todavía no tiene sentido: el resumen final (kappas por serie, kappas
  agrupados si corresponde y su lectura con los límites de §2.5 y §3.5) se escribe cuando estén
  las cuatro series.

---

# 6. Límites

- **Tamaño.** 24 decisiones y 4 pares en ES2016, 25 decisiones y 8 pares en IS1003: los
  porcentajes y los kappas de §2 y §3 tienen incertidumbre grande, y los de pares son indefinidos
  o inestables.
- **Solo agregados.** Al ser series de evaluación, el notebook no permite revisar por qué `jss`
  resolvió cada caso; esa información está en las razones de los CSV adjudicados, que no se
  reproducen aquí (ADR 0005).
- **Los kappas de estado son diagnósticos** y no se citan como acuerdo entre anotadores de OE1.
- **Una sola adjudicadora.** Que `jss` coincida más con uno de los anotadores no mide la
  calidad de ninguno: mide la distancia entre dos lecturas y la tercera.

## 6.1 Reproducibilidad

Todo lo mostrado sale de archivos versionados; no requiere el corpus AMI.

```bash
# Re-ejecutar este notebook de punta a punta
uv run --group notebooks jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=900 notebooks/18-jss-anotacion-f2-adjudicacion.ipynb

# Regenerar el acuerdo y verificar la adjudicación (insumos de este notebook)
uv run afg gold agreement --series ES2016
uv run afg gold validate-adjudication --series ES2016
uv run afg gold agreement --series IS1003
uv run afg gold validate-adjudication --series IS1003
```

Las funciones que calculan cada cifra están en
[`src/afg/annotation/calibration.py`](../src/afg/annotation/calibration.py) y
[`src/afg/annotation/agreement.py`](../src/afg/annotation/agreement.py), con sus pruebas en
`tests/test_calibration.py` y `tests/test_agreement.py`.